# Stage 2 — Stable v2 extended pre-training

This run initializes **only from the Stage 1 checkpoint** `best_model_fixed.weights.h5` (or `CEPHALO_FIXED_WEIGHTS`). It uses spatial-softmax/KL + smooth-L1, trains on the combined ISBI folders, and validates on the fixed 60-image in-house set. Its versioned output is not an input to Stage 3.

In [ ]:
import os
import sys
from pathlib import Path

REPO_ROOT = None
for start in (Path.cwd().resolve(), Path('/kaggle/working')):
    for candidate in (start, *start.parents):
        if (candidate / 'src' / 'config.py').is_file():
            REPO_ROOT = candidate
            break
    if REPO_ROOT is not None:
        break
if REPO_ROOT is None:
    raise FileNotFoundError('Could not locate repository root containing src/config.py')
SRC_DIR = REPO_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

def resolve_stage1_checkpoint():
    env_path = os.getenv('CEPHALO_FIXED_WEIGHTS')
    candidates = [Path(env_path).expanduser()] if env_path else []
    candidates += [
        REPO_ROOT / 'checkpoints' / 'best_model_fixed.weights.h5',
        Path('/kaggle/working/best_model_fixed.weights.h5'),
        Path('/kaggle/working/checkpoints/best_model_fixed.weights.h5'),
    ]
    kaggle_input = Path('/kaggle/input')
    if kaggle_input.is_dir():
        candidates += sorted(kaggle_input.rglob('best_model_fixed.weights.h5'))
    for candidate in candidates:
        if candidate.is_file():
            return str(candidate.resolve())
    raise FileNotFoundError('Stage 1 best_model_fixed.weights.h5 not found; set CEPHALO_FIXED_WEIGHTS')

ISBI_ROOT = os.getenv('CEPHALO_ISBI_ROOT', '/kaggle/input/cropped-isbi-dataset/cropped_isbi_dataset/ISBI Dataset')
FT_ROOT = os.getenv('CEPHALO_FT_ROOT', '/kaggle/input/fine-tuning-dataset-final') if Path('/kaggle').is_dir() else str(REPO_ROOT / 'data' / 'inhouse')
FIXED_WEIGHTS = resolve_stage1_checkpoint()
LOG_DIR = os.getenv('CEPHALO_LOG_DIR', '/kaggle/working/training_logs' if Path('/kaggle').is_dir() else str(REPO_ROOT / 'results'))
CHECKPOINT_DIR = os.getenv('CEPHALO_CHECKPOINT_DIR', '/kaggle/working/checkpoints' if Path('/kaggle').is_dir() else str(REPO_ROOT / 'checkpoints'))

from config import make_stage2_extended_config, STAGE3_VAL_IDS, resolve_data_dir
from reproducibility import set_global_seed
cfg = make_stage2_extended_config(dataset_root=ISBI_ROOT, inhouse_root=FT_ROOT, pretrained_weights=FIXED_WEIGHTS, log_dir=LOG_DIR, checkpoint_dir=CHECKPOINT_DIR)
set_global_seed(cfg.seed)  # before any data loader or model is created

assert Path(cfg.pretrained_weights).name == 'best_model_fixed.weights.h5'
assert 'extended' not in Path(cfg.pretrained_weights).name.lower()
assert cfg.objective_version == 'v2' and cfg.decoding_version == 'spatial_softmax'
assert cfg.epochs == 120 and cfg.learning_rate == 3e-5 and cfg.batch_size == 4
assert cfg.decoder_dropout_rate == 0.10 and cfg.gradient_clip_norm == 2.0
assert cfg.optimizer == 'adamw' and cfg.weight_decay == 1e-5
assert cfg.heatmap_loss == 'spatial_kl' and cfg.heatmap_loss_weight == 0.1
assert cfg.coord_loss == 'smooth_l1' and cfg.coord_loss_weight == 10.0 and cfg.smooth_l1_delta == 0.01
assert cfg.freeze_encoder_batch_norm and cfg.output_filename == 'best_model_stage2_v2.weights.h5'
val_image_ids = list(STAGE3_VAL_IDS)
assert len(val_image_ids) == 60 and len(set(val_image_ids)) == 60
print('Stage 1 initialization checkpoint:', cfg.pretrained_weights)
print('Versioned output:', Path(cfg.checkpoint_dir) / cfg.output_filename)

In [ ]:
from augmentation import CephalometricAugmentation
from data import ISBIPathDataset, InHouseDataset, CephalometricDataLoader, collect_isbi_image_paths
from model import build_model
from train import train

isbi_image_paths = collect_isbi_image_paths(cfg.dataset_root)
isbi_dataset = ISBIPathDataset(isbi_image_paths, cfg, Path(cfg.dataset_root) / 'Annotations')
train_loader = CephalometricDataLoader(isbi_dataset, cfg, shuffle=True, augmentation=CephalometricAugmentation(cfg))
if Path('/kaggle').is_dir():
    FT_IMAGES = resolve_data_dir(FT_ROOT, Path('images_v2') / 'images_v2')
    FT_ANNOTATIONS = resolve_data_dir(FT_ROOT, Path('annotations_v2') / 'annotations_v2')
else:
    FT_IMAGES, FT_ANNOTATIONS = str(Path(FT_ROOT) / 'images'), str(Path(FT_ROOT) / 'annotations')
val_loader = InHouseDataset(FT_ROOT, val_image_ids, cfg, training=False, shuffle=False, image_dir=FT_IMAGES, annotation_dir=FT_ANNOTATIONS)
model, encoder_layer_names = build_model(cfg)
# Do not load here: train() performs the single strict Stage 1 load and records it.
print('ISBI training images:', len(isbi_image_paths), '| validation images:', len(val_image_ids))
print('Model outputs:', model.output_shape)

In [ ]:
history = train(cfg, train_loader=train_loader, val_loader=val_loader, model=model, encoder_layer_names=encoder_layer_names)
assert history['incoming_checkpoint'] == cfg.pretrained_weights
print('Best epoch:', history['best_epoch'], '| best validation MRE:', history['best_val_mre'])
print('History:', Path(cfg.log_dir) / cfg.history_filename)
print('Best v2 checkpoint:', Path(cfg.checkpoint_dir) / cfg.output_filename)

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(8, 4))
plt.axhline(history['baseline']['mre'], color='gray', linestyle='--', label='epoch-0 Stage 1 baseline')
plt.plot(history['epoch'], history['train_mre'], label='train MRE')
plt.plot(history['epoch'], history['val_mre'], label='validation MRE')
plt.xlabel('Epoch'); plt.ylabel('MRE (mm)'); plt.title('Stage 2 stable-v2 history')
plt.legend(); plt.grid(alpha=0.25); plt.show()